# 04b - Pengujian Hipotesis
Materi Modul 4: hipotesis, tingkat signifikansi & nilai-p, kesalahan Tipe 1 vs Tipe 2, statistik F, korelasi vs sebab-akibat.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)
wine = pd.read_csv('../datasets/wine.csv')
bc = pd.read_csv('../datasets/breast_cancer.csv')
alpha = 0.05

## 1. Uji-t dua sampel
H0: rata-rata `mean radius` tumor ganas = jinak.  H1: berbeda.

In [ ]:
ganas = bc.loc[bc['target']==0, 'mean radius']
jinak = bc.loc[bc['target']==1, 'mean radius']
t, p = stats.ttest_ind(ganas, jinak, equal_var=False)
print(f'Welch t = {t:.3f}, p-value = {p:.2e}')
print('Keputusan:', 'Tolak H0' if p < alpha else 'Gagal tolak H0')

## 2. Alternatif non-parametrik (Mann-Whitney U)

In [ ]:
u, p_u = stats.mannwhitneyu(ganas, jinak)
print(f'U = {u:.1f}, p-value = {p_u:.2e}')

## 3. ANOVA & statistik F
H0: rata-rata `alcohol` sama di ketiga kelas wine.

In [ ]:
grup = [wine.loc[wine['target']==k, 'alcohol'] for k in (0,1,2)]
F, p_f = stats.f_oneway(*grup)
print(f'F = {F:.2f}, p-value = {p_f:.2e}')

## 4. Uji Chi-square (dua variabel kategorikal)

In [ ]:
wine['kadar_alkohol'] = pd.qcut(wine['alcohol'], 3, labels=['rendah','sedang','tinggi'])
kt = pd.crosstab(wine['kadar_alkohol'], wine['target'])
chi2, p_c, dof, _ = stats.chi2_contingency(kt)
print(kt); print(f'chi2 = {chi2:.2f}, dof = {dof}, p-value = {p_c:.2e}')

## 5. Kesalahan Tipe 1 vs Tipe 2 (simulasi)
- Tipe 1: menolak H0 padahal H0 benar (probabilitas = alpha)
- Tipe 2: gagal menolak H0 padahal H1 benar (power = 1 - Tipe 2)

In [ ]:
def tolak(mean_b, n=30, sims=2000):
    return np.mean([stats.ttest_ind(rng.normal(0,1,n), rng.normal(mean_b,1,n)).pvalue < alpha for _ in range(sims)])
print('H0 benar (selisih 0)    -> laju Tipe 1 :', tolak(0.0))
print('H1 benar (selisih 0.5)  -> power        :', tolak(0.5), '| Tipe 2 =', round(1-tolak(0.5),3))

## 6. Korelasi vs Sebab-akibat
Contoh klasik: penjualan es krim & kasus tenggelam sama-sama dipengaruhi suhu (confounder).

In [ ]:
suhu = rng.normal(28, 4, 300)
es_krim = 5*suhu + rng.normal(0, 10, 300)
tenggelam = 0.3*suhu + rng.normal(0, 2, 300)
print('Korelasi es krim vs tenggelam:', round(np.corrcoef(es_krim, tenggelam)[0,1], 3))
# kontrol suhu: korelasi residual
res_e = es_krim - np.polyval(np.polyfit(suhu, es_krim, 1), suhu)
res_t = tenggelam - np.polyval(np.polyfit(suhu, tenggelam, 1), suhu)
print('Setelah suhu dikontrol       :', round(np.corrcoef(res_e, res_t)[0,1], 3))

## Kesimpulan
- (ringkas keputusan tiap uji dan artinya; kenapa korelasi tidak sama dengan kausalitas)